# Loading data from Staging Dir

In [0]:

customer_df = spark.read.csv(path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/customers")

products_df = spark.read.json(path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/products")

orders_df = spark.read.json(path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/orders")

# Use Catalog and create schemas

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
spark.sql(f"USE CATALOG {catalog_name};")


In [0]:
%sql
-- USE CATALOG wns24082026;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;


# Bronze Layer

In [0]:
customer_df.write.saveAsTable("bronze.customer", mode="OVERWRITE")
products_df.write.saveAsTable("bronze.products", mode="OVERWRITE")
orders_df.write.saveAsTable("bronze.orders", mode="OVERWRITE")

In [0]:
spark.read.table("bronze.customer").display()
spark.read.table("bronze.products").display()
spark.read.table("bronze.orders").display()

# Silver Layer

In [0]:
from pyspark.sql.functions import col

spark.read.table("bronze.orders").filter(col("order_id").isNotNull()).withColumn(
    "total_price", col("qty") * col("price")
).write.saveAsTable("silver.orders", mode="OVERWRITE")

In [0]:
spark.read.table("silver.orders").display()

# Gold Layer

In [0]:
from pyspark.sql.functions import sum

spark.read.table("silver.orders").groupBy("item_id").agg(
    sum("total_price").alias("sum_total_price")
).write.saveAsTable("gold.revenue_by_product", mode="OVERWRITE")

In [0]:
spark.read.table("gold.revenue_by_product").display()